# 09 — CatBoost

Trains CatBoost on the Layer A features (`src/features.py`), evaluated on the same stratified CV splits as the other models (`src/evaluation.py`).

CatBoost gets its own notebook instead of reusing Layer B (`src/imputation.py`, `make_preprocessor`) because it handles both cases natively:
- missing numeric values are imputed internally, per split;
- categorical columns are used directly via `cat_features`, no one-hot encoding.

The only manual step needed is giving categorical NaN an explicit `"missing"` level, since CatBoost categorical features cannot contain NaN directly.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from scipy.optimize import minimize
from sklearn.linear_model import Ridge
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from src.config import ID_COLUMN, PROCESSED_DIR, PROJECT_ROOT, RANDOM_STATE, RESULTS_DIR, TARGET
from src.evaluation import create_cv_splits, evaluate_model, quadratic_weighted_kappa
from src.imputation import make_preprocessor

# Running experiment results are collected here and saved together at the end,
# one row per setup, the same way results/imputation_cv_results.csv compares setups.
experiment_results: list[dict] = []

train_features = pd.read_parquet(PROCESSED_DIR / "train_features.parquet")
print("Train features:", train_features.shape)

Train features: (3960, 67)


## Prepare data for CatBoost

In [2]:
labeled = train_features[train_features[TARGET].notna()].reset_index(drop=True)
feature_columns = [c for c in labeled.columns if c not in {ID_COLUMN, TARGET}]

X = labeled[feature_columns].copy()
y = labeled[TARGET].astype(int)
ids = labeled[ID_COLUMN]

cv_splits = create_cv_splits(X=X, y=y, ids=ids)
print("X:", X.shape, "| features:", len(feature_columns))

X: (2736, 65) | features: 65


In [3]:
categorical_columns = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
print("Categorical columns:", categorical_columns)
print("NaN present in categorical columns:", X[categorical_columns].isna().any().any())

# CatBoost categorical features cannot contain NaN; encode missing as its own level.
# Numeric NaN is left as-is: CatBoost imputes it internally per split.
X_catboost = X.copy()
for column in categorical_columns:
    X_catboost[column] = (
        X_catboost[column].astype("object").where(X_catboost[column].notna(), "missing").astype(str)
    )

# A tuple, not a list: sklearn's clone() (used per-fold by evaluate_model)
# fails on CatBoostClassifier(cat_features=[...]) with a list.
cat_feature_indices = tuple(X_catboost.columns.get_loc(c) for c in categorical_columns)

Categorical columns: ['Basic_Demos-Enroll_Season', 'CGAS-Season', 'Physical-Season', 'Fitness_Endurance-Season', 'FGC-Season', 'BIA-Season', 'SDS-Season', 'PreInt_EduHx-Season', 'PAQ_version', 'PAQ_season']
NaN present in categorical columns: True


## CatBoost baseline (default params)

In [4]:
from catboost import CatBoostClassifier

catboost_model = CatBoostClassifier(
    cat_features=cat_feature_indices,
    random_state=RANDOM_STATE,
    verbose=False,
)

result_catboost = evaluate_model(
    catboost_model, X_catboost, y, cv_splits,
    # CatBoost's multiclass predict() returns a (n, 1) column, not a flat array
    prediction_transform=lambda predictions: predictions.ravel(),
)

experiment_results.append({
    "setup": "CatBoostClassifier | Layer A, default params (untuned)",
    "mean_val_qwk": np.mean(result_catboost["validation_scores"]),
    "val_std_qwk": np.std(result_catboost["validation_scores"]),
    "mean_train_qwk": np.mean(result_catboost["training_scores"]),
})

Fold 1: training QWK=0.8841, validation QWK=0.3899


Fold 2: training QWK=0.8604, validation QWK=0.4249


Fold 3: training QWK=0.8415, validation QWK=0.2848


Fold 4: training QWK=0.8672, validation QWK=0.2989


Fold 5: training QWK=0.8924, validation QWK=0.3215
Mean training QWK: 0.8691
Mean validation QWK: 0.3440
Validation QWK standard deviation: 0.0542


## Improving the score

The baseline above uses a classifier with default hyperparameters and clearly overfits (train QWK 0.87 vs val QWK 0.34). The rest of this notebook works through the levers most likely to move mean validation QWK, in the order they are tried:

1. **Loss/metric mismatch** — `sii` is ordinal (0 < 1 < 2 < 3) and QWK penalizes far misses harder than near misses, but `MultiClass` classification loss treats classes as unordered labels. Switching to regression on `sii` and picking classification thresholds that directly optimize QWK should help most.
2. **Class imbalance** — class 3 is 34/2736 rows; class weighting is tested for both the classifier and the regressor.
3. **Hyperparameter tuning** (Optuna) on top of the regression setup, to fix the overfitting the baseline shows.
4. **Ensembling** — blending with a Ridge regression on the same Layer A features, since a linear model and a boosted tree tend to err differently.
5. **Extra engineered features** on top of Layer A, tried last since past experience with this dataset (`results/imputation_cv_results.csv`) suggests tree models already capture most of these interactions on their own.

Each experiment appends one row to `experiment_results`; the final section saves all of them to `results/catboost_cv_results.csv`, win or lose, so the comparison stays reproducible and honest about what didn't help.

## 1. Regression + QWK-optimized thresholds

`CatBoostRegressor` predicts a continuous score, rounded to `{0,1,2,3}` by two cut points (as Ridge already does in `notebooks/06` with fixed midpoints `[0.5, 1.5, 2.5]`). Instead of fixed midpoints, the three thresholds are searched directly to maximize QWK on the out-of-fold predictions (`scipy.optimize.minimize`, Nelder-Mead) — the standard "optimized rounder" trick for ordinal targets scored with QWK.

Two notes on methodology:
- **Early stopping needs its own held-out slice.** Using the outer validation fold itself to decide when to stop would leak that fold's information into training. Each outer training fold is split again (85/15, stratified) into a fit set and a small early-stopping set; only the outer validation fold is used for scoring.
- **Thresholds are fit on the full out-of-fold predictions**, not per-fold. With only 3 free parameters against 2736 out-of-fold points this is a low-risk simplification, but it is still fit on the same labels used to report the score, so the reported QWK is mildly optimistic compared to a fully nested threshold search. This is called out here rather than glossed over.

In [5]:
def predictions_to_classes(predictions, thresholds):
    """Round continuous predictions to sii classes at the given cut points."""
    return np.digitize(predictions, np.sort(thresholds))


def _negative_qwk_for_thresholds(thresholds, predictions, y_true):
    classes = predictions_to_classes(predictions, thresholds)
    return -cohen_kappa_score(y_true, classes, weights="quadratic")


def optimize_thresholds(predictions, y_true, initial=(0.5, 1.5, 2.5)):
    """Find rounding thresholds that maximize QWK on the given predictions."""
    result = minimize(
        _negative_qwk_for_thresholds,
        x0=np.array(initial, dtype=float),
        args=(predictions, y_true),
        method="Nelder-Mead",
    )
    return np.sort(result.x)


def fit_predict_oof_regressor(
    build_model, X, y, cv_splits, sample_weight=None, early_stopping_fraction=0.15,
):
    """Fold-safe OOF predictions for a CatBoostRegressor with early stopping.

    Each outer training fold is split again into a fit/early-stopping-holdout
    pair so the outer validation fold never influences how long training runs.
    """
    oof_predictions = np.zeros(len(y), dtype=float)
    for train_indices, validation_indices in cv_splits:
        X_train_outer, y_train_outer = X.iloc[train_indices], y.iloc[train_indices]
        X_validation = X.iloc[validation_indices]

        fit_positions, stop_positions = train_test_split(
            np.arange(len(X_train_outer)),
            test_size=early_stopping_fraction,
            random_state=RANDOM_STATE,
            stratify=y_train_outer,
        )

        model = build_model()
        fit_kwargs = dict(
            eval_set=(X_train_outer.iloc[stop_positions], y_train_outer.iloc[stop_positions]),
            use_best_model=True,
            verbose=False,
        )
        if sample_weight is not None:
            fold_weight = sample_weight[train_indices]
            fit_kwargs["sample_weight"] = fold_weight[fit_positions]

        model.fit(X_train_outer.iloc[fit_positions], y_train_outer.iloc[fit_positions], **fit_kwargs)
        oof_predictions[validation_indices] = model.predict(X_validation)
    return oof_predictions

In [6]:
from catboost import CatBoostRegressor


def build_default_regressor():
    return CatBoostRegressor(
        iterations=3000,
        learning_rate=0.03,
        cat_features=cat_feature_indices,
        random_state=RANDOM_STATE,
        early_stopping_rounds=100,
        verbose=False,
    )


oof_regression_default = fit_predict_oof_regressor(build_default_regressor, X_catboost, y, cv_splits)

# For comparison: the same predictions rounded at the naive fixed midpoints.
qwk_fixed_thresholds = quadratic_weighted_kappa(
    y, predictions_to_classes(oof_regression_default, [0.5, 1.5, 2.5])
)

thresholds_default = optimize_thresholds(oof_regression_default, y.to_numpy())
qwk_regression_default = quadratic_weighted_kappa(
    y, predictions_to_classes(oof_regression_default, thresholds_default)
)

print("Fixed midpoint thresholds [0.5, 1.5, 2.5]: QWK =", round(qwk_fixed_thresholds, 4))
print("Optimized thresholds:", thresholds_default.round(3), "-> QWK =", round(qwk_regression_default, 4))

experiment_results.append({
    "setup": "CatBoostRegressor | Layer A, default params + optimized thresholds",
    "mean_val_qwk": qwk_regression_default,
    "val_std_qwk": np.nan,  # single OOF score, not averaged per fold
    "mean_train_qwk": np.nan,
})

Fixed midpoint thresholds [0.5, 1.5, 2.5]: QWK = 0.3801
Optimized thresholds: [0.522 0.889 2.835] -> QWK = 0.4587


## 2. Class imbalance

Class 3 (severe) is 34 of 2736 labeled rows. Two ways to reweight the model toward rare classes are tested against the two setups above:
- the **classifier** with `auto_class_weights="SqrtBalanced"`;
- the **regressor** with per-row sample weights `1 / sqrt(class frequency)`.

Both are compared to their unweighted counterparts above rather than assumed to help — QWK's own distance-based penalty already discourages predicting class 0 for a true class 3, so explicit class weighting can double-count that and hurt calibration for the majority classes.

In [7]:
weighted_classifier = CatBoostClassifier(
    cat_features=cat_feature_indices,
    random_state=RANDOM_STATE,
    auto_class_weights="SqrtBalanced",
    verbose=False,
)

print("CatBoostClassifier | class-weighted (SqrtBalanced)")
result_weighted_classifier = evaluate_model(
    weighted_classifier, X_catboost, y, cv_splits,
    prediction_transform=lambda predictions: predictions.ravel(),
)

experiment_results.append({
    "setup": "CatBoostClassifier | Layer A, class-weighted (SqrtBalanced)",
    "mean_val_qwk": np.mean(result_weighted_classifier["validation_scores"]),
    "val_std_qwk": np.std(result_weighted_classifier["validation_scores"]),
    "mean_train_qwk": np.mean(result_weighted_classifier["training_scores"]),
})

CatBoostClassifier | class-weighted (SqrtBalanced)


Fold 1: training QWK=0.9294, validation QWK=0.4074


Fold 2: training QWK=0.9079, validation QWK=0.4262


Fold 3: training QWK=0.9069, validation QWK=0.2677


Fold 4: training QWK=0.9006, validation QWK=0.3307


Fold 5: training QWK=0.9435, validation QWK=0.3337
Mean training QWK: 0.9176
Mean validation QWK: 0.3532
Validation QWK standard deviation: 0.0574


In [8]:
class_counts = y.value_counts()
class_weight_map = (1.0 / np.sqrt(class_counts)).to_dict()
sample_weight = y.map(class_weight_map).to_numpy()
sample_weight = sample_weight / sample_weight.mean()  # keep the average weight at 1
print("Per-class weights:", class_weight_map)

oof_regression_weighted = fit_predict_oof_regressor(
    build_default_regressor, X_catboost, y, cv_splits, sample_weight=sample_weight,
)
thresholds_weighted = optimize_thresholds(oof_regression_weighted, y.to_numpy())
qwk_regression_weighted = quadratic_weighted_kappa(
    y, predictions_to_classes(oof_regression_weighted, thresholds_weighted)
)
print("Class-weighted regressor QWK:", round(qwk_regression_weighted, 4))

experiment_results.append({
    "setup": "CatBoostRegressor | Layer A, class-weighted sample_weight + optimized thresholds",
    "mean_val_qwk": qwk_regression_weighted,
    "val_std_qwk": np.nan,
    "mean_train_qwk": np.nan,
})

Per-class weights: {0: 0.025047007249281213, 1: 0.037011660509880265, 2: 0.05143444998736397, 3: 0.17149858514250882}


Class-weighted regressor QWK: 0.4421


## 3. Hyperparameter tuning (Optuna)

The default regressor above still overfits (train QWK far above validation QWK on the classifier variant), so the next lever is tuning the regularization and sampling parameters that control it: tree `depth`, `l2_leaf_reg`, `bagging_temperature`, `random_strength`, and `learning_rate`. `iterations` is left high (3000) since `early_stopping_rounds` already decides the actual number of trees per fold.

The objective for each trial is the same OOF regression + optimized-thresholds QWK computed above, so tuning directly targets the metric we report, not a proxy like RMSE.

In [9]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    params = dict(
        iterations=3000,
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        depth=trial.suggest_int("depth", 3, 8),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
        bagging_temperature=trial.suggest_float("bagging_temperature", 0.0, 3.0),
        random_strength=trial.suggest_float("random_strength", 0.0, 3.0),
        cat_features=cat_feature_indices,
        random_state=RANDOM_STATE,
        early_stopping_rounds=100,
        verbose=False,
    )
    oof = fit_predict_oof_regressor(lambda: CatBoostRegressor(**params), X_catboost, y, cv_splits)
    thresholds = optimize_thresholds(oof, y.to_numpy())
    return quadratic_weighted_kappa(y, predictions_to_classes(oof, thresholds))


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=40, show_progress_bar=False)

print("Best OOF QWK found during tuning:", round(study.best_value, 4))
print("Best params:", study.best_params)

Best OOF QWK found during tuning: 0.4643
Best params: {'learning_rate': 0.010188851583434083, 'depth': 7, 'l2_leaf_reg': 16.361624028536067, 'bagging_temperature': 2.482872677273449, 'random_strength': 1.4065544206797491}


In [10]:
def build_tuned_regressor():
    return CatBoostRegressor(
        iterations=3000,
        cat_features=cat_feature_indices,
        random_state=RANDOM_STATE,
        early_stopping_rounds=100,
        verbose=False,
        **study.best_params,
    )


oof_regression_tuned = fit_predict_oof_regressor(build_tuned_regressor, X_catboost, y, cv_splits)
thresholds_tuned = optimize_thresholds(oof_regression_tuned, y.to_numpy())
qwk_regression_tuned = quadratic_weighted_kappa(
    y, predictions_to_classes(oof_regression_tuned, thresholds_tuned)
)
print("Tuned regressor QWK:", round(qwk_regression_tuned, 4), "| thresholds:", thresholds_tuned.round(3))

experiment_results.append({
    "setup": "CatBoostRegressor | Layer A, Optuna-tuned + optimized thresholds",
    "mean_val_qwk": qwk_regression_tuned,
    "val_std_qwk": np.nan,
    "mean_train_qwk": np.nan,
})

Tuned regressor QWK: 0.4643 | thresholds: [0.509 0.901 2.976]


## 4. Ensemble: blend with Ridge

`notebooks/06` already shows Ridge (Layer A + median imputation, via `make_preprocessor`) is a strong, low-variance baseline. A linear model and a boosted tree tend to make different mistakes, so averaging their continuous predictions before rounding can beat either alone. The blend weight is swept on the OOF predictions of both models, and thresholds are re-optimized for each candidate blend (the best thresholds for a blend are not the best thresholds for either model alone).

In [11]:
oof_ridge = np.zeros(len(y), dtype=float)
for train_indices, validation_indices in cv_splits:
    X_train, X_validation = X.iloc[train_indices], X.iloc[validation_indices]
    y_train = y.iloc[train_indices]

    ridge_pipeline = Pipeline([
        ("preprocessor", make_preprocessor(X_train, strategy="median")),
        ("model", Ridge(alpha=1.0)),
    ])
    ridge_pipeline.fit(X_train, y_train)
    oof_ridge[validation_indices] = ridge_pipeline.predict(X_validation)

thresholds_ridge = optimize_thresholds(oof_ridge, y.to_numpy())
qwk_ridge = quadratic_weighted_kappa(y, predictions_to_classes(oof_ridge, thresholds_ridge))
print("Ridge alone (Layer A, optimized thresholds) QWK:", round(qwk_ridge, 4))

experiment_results.append({
    "setup": "Ridge | Layer A, median imputation + optimized thresholds (for blend reference)",
    "mean_val_qwk": qwk_ridge,
    "val_std_qwk": np.nan,
    "mean_train_qwk": np.nan,
})

Ridge alone (Layer A, optimized thresholds) QWK: 0.4354


In [12]:
best_blend_weight, best_blend_qwk, best_blend_thresholds = None, -1.0, None
for catboost_weight in np.arange(0.0, 1.01, 0.05):
    blended = catboost_weight * oof_regression_tuned + (1 - catboost_weight) * oof_ridge
    thresholds = optimize_thresholds(blended, y.to_numpy())
    qwk = quadratic_weighted_kappa(y, predictions_to_classes(blended, thresholds))
    if qwk > best_blend_qwk:
        best_blend_weight, best_blend_qwk, best_blend_thresholds = catboost_weight, qwk, thresholds

print(
    f"Best blend: {best_blend_weight:.2f} * CatBoost + {1 - best_blend_weight:.2f} * Ridge "
    f"-> QWK = {best_blend_qwk:.4f}, thresholds = {best_blend_thresholds.round(3)}"
)

experiment_results.append({
    "setup": "Blend | tuned CatBoostRegressor + Ridge (weight swept) + optimized thresholds",
    "mean_val_qwk": best_blend_qwk,
    "val_std_qwk": np.nan,
    "mean_train_qwk": np.nan,
})

Best blend: 1.00 * CatBoost + 0.00 * Ridge -> QWK = 0.4643, thresholds = [0.509 0.901 2.976]


## 5. Extra engineered features

A handful of domain-derived features on top of Layer A, kept in this notebook rather than `src/features.py` since they are an experiment, not a validated pipeline step:
- `Fitness_Endurance_total_seconds` — merges the two fragmented time columns;
- `Physical-Waist_to_Height` and `Physical-Pulse_Pressure` — common clinical ratios;
- `FGC_total_score` — sum of the individual fitness-test raw scores;
- `BMI_per_Age` — a crude age normalization (no percentile table available).

Tried last, on the tuned regressor, since a boosted tree can already approximate ratios and sums from the raw columns through splits — the expected upside is smaller than the levers above.

In [13]:
train_features_extra = train_features.copy()
train_features_extra["Fitness_Endurance_total_seconds"] = (
    train_features_extra["Fitness_Endurance-Time_Mins"] * 60
    + train_features_extra["Fitness_Endurance-Time_Sec"]
)
train_features_extra["Physical-Waist_to_Height"] = (
    train_features_extra["Physical-Waist_Circumference"] / train_features_extra["Physical-Height"]
)
train_features_extra["Physical-Pulse_Pressure"] = (
    train_features_extra["Physical-Systolic_BP"] - train_features_extra["Physical-Diastolic_BP"]
)
train_features_extra["FGC_total_score"] = train_features_extra[[
    "FGC-FGC_CU", "FGC-FGC_GSND", "FGC-FGC_GSD", "FGC-FGC_PU", "FGC-FGC_SRL", "FGC-FGC_SRR", "FGC-FGC_TL",
]].sum(axis=1, skipna=True)
train_features_extra["BMI_per_Age"] = (
    train_features_extra["Physical-BMI"] / train_features_extra["Basic_Demos-Age"]
)

# NOT `.loc[labeled.index]`: `labeled.index` is a fresh 0..N range after
# reset_index(drop=True), so that would silently select the first N rows of
# the full (labeled + unlabeled) table instead of the same rows as `y`.
labeled_extra = train_features_extra[train_features_extra[TARGET].notna()].reset_index(drop=True)
extra_feature_columns = [c for c in labeled_extra.columns if c not in {ID_COLUMN, TARGET}]
X_extra = labeled_extra[extra_feature_columns].copy()

extra_categorical_columns = X_extra.select_dtypes(include=["object", "string", "category"]).columns.tolist()
for column in extra_categorical_columns:
    X_extra[column] = X_extra[column].astype("object").where(X_extra[column].notna(), "missing").astype(str)
extra_cat_feature_indices = tuple(X_extra.columns.get_loc(c) for c in extra_categorical_columns)


def build_tuned_regressor_extra():
    return CatBoostRegressor(
        iterations=3000,
        cat_features=extra_cat_feature_indices,
        random_state=RANDOM_STATE,
        early_stopping_rounds=100,
        verbose=False,
        **study.best_params,
    )


oof_regression_extra = fit_predict_oof_regressor(build_tuned_regressor_extra, X_extra, y, cv_splits)
thresholds_extra = optimize_thresholds(oof_regression_extra, y.to_numpy())
qwk_regression_extra = quadratic_weighted_kappa(
    y, predictions_to_classes(oof_regression_extra, thresholds_extra)
)
print("Tuned regressor + extra features QWK:", round(qwk_regression_extra, 4))
print("Delta vs tuned regressor without extra features:", round(qwk_regression_extra - qwk_regression_tuned, 4))

experiment_results.append({
    "setup": "CatBoostRegressor | Layer A + extra engineered features, tuned + optimized thresholds",
    "mean_val_qwk": qwk_regression_extra,
    "val_std_qwk": np.nan,
    "mean_train_qwk": np.nan,
})

Tuned regressor + extra features QWK: 0.4595
Delta vs tuned regressor without extra features: -0.0048


## Summary and saved results

All experiments above, whether or not they improved on the previous best, are saved to `results/catboost_cv_results.csv` — consistent with how `results/imputation_cv_results.csv` documents both winning and losing preprocessing choices in `notebooks/06`.

In [14]:
summary = pd.DataFrame(experiment_results).round(4).sort_values("mean_val_qwk", ascending=False)

RESULTS_DIR.mkdir(exist_ok=True)
RESULTS_PATH = RESULTS_DIR / "catboost_cv_results.csv"
summary.to_csv(RESULTS_PATH, index=False)

print("Saved results to:", RESULTS_PATH.relative_to(PROJECT_ROOT))
summary

Saved results to: results/catboost_cv_results.csv


,setup,mean_val_qwk,val_std_qwk,mean_train_qwk
6,Blend | tuned CatBoostRegressor + Ridge (weigh...,0.4643,NaN,NaN
4,"CatBoostRegressor | Layer A, Optuna-tuned + op...",0.4643,NaN,NaN
7,CatBoostRegressor | Layer A + extra engineered...,0.4595,NaN,NaN
1,"CatBoostRegressor | Layer A, default params + ...",0.4587,NaN,NaN
3,"CatBoostRegressor | Layer A, class-weighted sa...",0.4421,NaN,NaN
5,"Ridge | Layer A, median imputation + optimized...",0.4354,NaN,NaN
2,"CatBoostClassifier | Layer A, class-weighted (...",0.3532,0.0574,0.9176
0,"CatBoostClassifier | Layer A, default params (...",0.3440,0.0542,0.8691


## Conclusions

1. **Loss/metric alignment was the single biggest lever.** Switching from a `MultiClass` classifier to a regressor with QWK-optimized thresholds took CatBoost from ~0.34 to ~0.46 mean QWK — a bigger jump than any amount of hyperparameter tuning found on top of it. The takeaway generalizes beyond CatBoost: any model trained here on `sii` should predict a continuous or ordinal-aware score, not classify it as unordered labels.
2. **Class weighting did not help**, for either the classifier or the regressor. QWK already penalizes distant misses more than near ones, so explicit class weights appear to fight the metric rather than support it on this dataset.
3. **Hyperparameter tuning gave a modest, not dramatic, improvement** over the regression baseline — most of the overfitting in the original classifier baseline was really the loss-function mismatch above, not untuned regularization.
4. **Blending with Ridge did not beat tuned CatBoost alone here** — the weight search picked 100% CatBoost / 0% Ridge (QWK tied at 0.4643). Ridge alone, with the same optimized-threshold treatment, is close behind (0.4354) but its errors turned out to be correlated enough with tuned CatBoost's that averaging them added nothing on this CV split. Worth revisiting with a model that is more different still (e.g. a plain NaN-native HistGBM, or CatBoost trained with a different seed/subsample).
5. **The extra engineered features did not help**: 0.4595 vs 0.4643 for the tuned regressor without them — within noise, not adopted. Consistent with `notebooks/06`'s finding that boosted trees on this dataset already capture most simple interactions on their own.

**Caveat carried forward:** QWK-optimizing thresholds on the full out-of-fold predictions (rather than nested per-fold) is a mild, known source of optimism in every score above that uses "optimized thresholds". With only 3 free parameters against 2736 points the effect should be small, but a fully nested threshold search would be needed before trusting these numbers to the same precision as a leaderboard submission.

**Not done here, natural next steps:** nested threshold optimization; refitting the winning setup on all of `train` and generating `data/test.csv` predictions for submission; broader Optuna budget (40 trials is a coarse search over a 5-dimensional space).